# Gold 3_19 — Customer Engagement Versions (SCD2)

One row per lead per **engagement version** — a new version opens only when the engagement hash changes. Complements 3_15 (daily snapshots = trends) with a true change log (versions = what changed, and when).

* **Population**: all Coaching Client leads from `gold.health_metrics` (intern clients included — currently 1,339).
* **Hash** (opens a version on change): LEAD_ID, LEAD_EMAIL, LEAD_NAME, CONTRACT_VALUE, SENTIMENTS_LAST_30_DAYS, SLACK_ENGAGED_FLAG, PLATFORM_ENGAGED_FLAG, MEETING_ENGAGED_FLAG, ENGAGEMENT_PATTERN, BASE_ENGAGEMENT_SCORE, FINAL_SCORE, HEALTH_BAND.
* **VRSN_START_DATE** = run date the version opened (first run = today). **VRSN_END_DATE** = run date the hash changed (the version was superseded); NULL while current.
* Clock metrics (DAYS_SINCE_*, TOTAL_MISSED_CALLS, CONTRACT_START_DATE) are **not hashed** — they refresh on the open version every run; closed versions keep their as-of values.
* TOTAL_MISSED_CALLS is the trailing **14-day window** count, not a lifetime total. It still influences versions through FINAL_SCORE / HEALTH_BAND; only score-neutral call changes (e.g. 1→2) do not open one.
* A lead that leaves the Coaching Client population gets its open version closed that run; if it returns, a new version opens.
* SOURCE_UPDATED_AT = the CRM lead record's `date_updated`. INSERT_DATE = when this pipeline created the version row.
* Scoring identical to 3_14 (spec 5.1): base from pattern (90, 80, 75, 60, 50, 40, 30, 20, or no-data), FINAL_SCORE = base + missed-call adjustment + sentiment adjustment — no adjustments on NO DATA, not capped.

In [0]:
%sql
-- Engagement source: the 3_14 scoring logic applied to the FULL health_metrics
-- population (intern clients included — this is the versioned record of everyone),
-- plus the change-detection hash. Hash = engagement state + identity + contract
-- value; clock metrics deliberately excluded (they drift daily and would
-- degenerate SCD2 into a daily snapshot — 3_15 already owns that job).
CREATE OR REPLACE TEMP VIEW customer_engagement_source AS (
WITH base AS (
    SELECT
        h.lead_id                                                          AS LEAD_ID,
        h.lead_name                                                        AS LEAD_NAME,
        h.primary_customer_email                                           AS LEAD_EMAIL,
        TRY_CAST(REGEXP_REPLACE(h.Contracted_Value, '[^0-9.]', '') AS DOUBLE) AS CONTRACT_VALUE,
        TRY_CAST(h.date_of_sale AS DATE)                                   AS CONTRACT_START_DATE,
        h.days_since_last_email                                            AS DAYS_SINCE_LAST_EMAIL,
        h.days_since_last_login                                            AS DAYS_SINCE_LAST_LOGIN,
        h.days_since_last_meeting                                          AS DAYS_SINCE_LAST_MEETING,
        h.days_since_last_message_from_client                              AS DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT,
        h.days_since_last_message_from_team_member                         AS DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER,
        h.missed_calls_14_days                                             AS TOTAL_MISSED_CALLS,
        COALESCE(UPPER(TRIM(h.sentiment)), 'NO_SENTIMENT')                 AS SENTIMENTS_LAST_30_DAYS,
        TRY_CAST(h.date_updated AS TIMESTAMP)                              AS SOURCE_UPDATED_AT,
        CASE WHEN h.days_since_last_message_from_client IS NULL THEN NULL
             WHEN h.days_since_last_message_from_client <= 14 THEN 1
             ELSE 0 END                                                    AS SLACK_ENGAGED_FLAG,
        CASE WHEN h.days_since_last_login IS NULL THEN NULL
             WHEN h.days_since_last_login <= 14 THEN 1
             ELSE 0 END                                                    AS PLATFORM_ENGAGED_FLAG,
        CASE WHEN h.days_since_last_meeting IS NULL OR h.days_since_last_meeting_calendly IS NULL THEN NULL
             WHEN h.days_since_last_meeting <= 30 OR h.days_since_last_meeting_calendly <= 30 THEN 1
             ELSE 0 END                                                    AS MEETING_ENGAGED_FLAG,
        CASE WHEN h.missed_calls_14_days = 0 THEN 5
             WHEN h.missed_calls_14_days BETWEEN 1 AND 2 THEN 0
             WHEN h.missed_calls_14_days >= 3 THEN -15 END                 AS missed_call_adjustment,
        CASE WHEN UPPER(TRIM(h.sentiment)) = 'POSITIVE' THEN 10
             WHEN UPPER(TRIM(h.sentiment)) = 'NEUTRAL'  THEN 5
             WHEN UPPER(TRIM(h.sentiment)) = 'NEGATIVE' THEN -25
             ELSE 0 END                                                    AS sentiment_adjustment
    FROM crm_ingestion.gold.health_metrics h
),
patterns AS (
    SELECT
        b.*,
        CASE
            WHEN b.SLACK_ENGAGED_FLAG = 1 AND b.PLATFORM_ENGAGED_FLAG = 1 AND b.MEETING_ENGAGED_FLAG = 1 THEN 'SLACK + PLATFORM + MEETING'
            WHEN b.SLACK_ENGAGED_FLAG = 1 AND b.PLATFORM_ENGAGED_FLAG = 1 AND (b.MEETING_ENGAGED_FLAG = 0 OR b.MEETING_ENGAGED_FLAG IS NULL) THEN 'SLACK + PLATFORM'
            WHEN (b.SLACK_ENGAGED_FLAG = 0 OR b.SLACK_ENGAGED_FLAG IS NULL) AND b.PLATFORM_ENGAGED_FLAG = 1 AND b.MEETING_ENGAGED_FLAG = 1 THEN 'PLATFORM + MEETING'
            WHEN b.SLACK_ENGAGED_FLAG = 1 AND (b.PLATFORM_ENGAGED_FLAG = 0 OR b.PLATFORM_ENGAGED_FLAG IS NULL) AND b.MEETING_ENGAGED_FLAG = 1 THEN 'SLACK + MEETING'
            WHEN b.SLACK_ENGAGED_FLAG = 1 AND (b.PLATFORM_ENGAGED_FLAG = 0 OR b.PLATFORM_ENGAGED_FLAG IS NULL) AND (b.MEETING_ENGAGED_FLAG = 0 OR b.MEETING_ENGAGED_FLAG IS NULL) THEN 'SLACK ONLY'
            WHEN (b.SLACK_ENGAGED_FLAG = 0 OR b.SLACK_ENGAGED_FLAG IS NULL) AND b.PLATFORM_ENGAGED_FLAG = 1 AND (b.MEETING_ENGAGED_FLAG = 0 OR b.MEETING_ENGAGED_FLAG IS NULL) THEN 'PLATFORM ONLY'
            WHEN (b.SLACK_ENGAGED_FLAG = 0 OR b.SLACK_ENGAGED_FLAG IS NULL) AND (b.PLATFORM_ENGAGED_FLAG = 0 OR b.PLATFORM_ENGAGED_FLAG IS NULL) AND b.MEETING_ENGAGED_FLAG = 1 THEN 'MEETING ONLY'
            WHEN b.SLACK_ENGAGED_FLAG = 0 AND b.PLATFORM_ENGAGED_FLAG = 0 AND b.MEETING_ENGAGED_FLAG = 0 THEN 'NO ENGAGEMENT'
            WHEN b.SLACK_ENGAGED_FLAG = 0 AND b.PLATFORM_ENGAGED_FLAG = 0 AND b.MEETING_ENGAGED_FLAG IS NULL THEN 'NO ENGAGEMENT'
            WHEN b.SLACK_ENGAGED_FLAG IS NULL AND b.PLATFORM_ENGAGED_FLAG = 0 AND b.MEETING_ENGAGED_FLAG = 0 THEN 'NO ENGAGEMENT'
            WHEN b.SLACK_ENGAGED_FLAG = 0 AND b.PLATFORM_ENGAGED_FLAG IS NULL AND b.MEETING_ENGAGED_FLAG = 0 THEN 'NO ENGAGEMENT'
            ELSE 'NO DATA'
        END AS ENGAGEMENT_PATTERN
    FROM base b
),
scored AS (
    SELECT
        p.*,
        CASE p.ENGAGEMENT_PATTERN
            WHEN 'SLACK + PLATFORM + MEETING' THEN 90
            WHEN 'SLACK + PLATFORM'         THEN 80
            WHEN 'PLATFORM + MEETING'       THEN 75
            WHEN 'SLACK + MEETING'          THEN 60
            WHEN 'PLATFORM ONLY'            THEN 50
            WHEN 'MEETING ONLY'            THEN 40
            WHEN 'SLACK ONLY'              THEN 30
            WHEN 'NO ENGAGEMENT'           THEN 20
            ELSE NULL
        END AS BASE_ENGAGEMENT_SCORE
    FROM patterns p
),
finals AS (
    SELECT
        s.*,
        CASE WHEN s.ENGAGEMENT_PATTERN = 'NO DATA' THEN NULL
             ELSE s.BASE_ENGAGEMENT_SCORE + s.missed_call_adjustment + s.sentiment_adjustment
        END AS FINAL_SCORE,
        CASE WHEN s.ENGAGEMENT_PATTERN = 'NO DATA' THEN 'NO DATA'
             WHEN s.BASE_ENGAGEMENT_SCORE + s.missed_call_adjustment + s.sentiment_adjustment >= 75 THEN 'GOOD'
             WHEN s.BASE_ENGAGEMENT_SCORE + s.missed_call_adjustment + s.sentiment_adjustment >= 50 THEN 'AVERAGE'
             ELSE 'POOR'
        END AS HEALTH_BAND
    FROM scored s
)
SELECT
    f.LEAD_ID, f.LEAD_EMAIL, f.LEAD_NAME,
    f.CONTRACT_VALUE, f.CONTRACT_START_DATE,
    f.DAYS_SINCE_LAST_EMAIL, f.DAYS_SINCE_LAST_LOGIN, f.DAYS_SINCE_LAST_MEETING,
    f.DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT, f.DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER,
    f.TOTAL_MISSED_CALLS, f.SENTIMENTS_LAST_30_DAYS,
    f.SLACK_ENGAGED_FLAG, f.PLATFORM_ENGAGED_FLAG, f.MEETING_ENGAGED_FLAG,
    f.ENGAGEMENT_PATTERN, f.BASE_ENGAGEMENT_SCORE, f.FINAL_SCORE, f.HEALTH_BAND,
    f.SOURCE_UPDATED_AT,
    MD5(TO_JSON(STRUCT(
        f.LEAD_ID, f.LEAD_EMAIL, f.LEAD_NAME,
        f.CONTRACT_VALUE, f.SENTIMENTS_LAST_30_DAYS,
        f.SLACK_ENGAGED_FLAG, f.PLATFORM_ENGAGED_FLAG, f.MEETING_ENGAGED_FLAG,
        f.ENGAGEMENT_PATTERN, f.BASE_ENGAGEMENT_SCORE, f.FINAL_SCORE, f.HEALTH_BAND
    ))) AS HASH_VALUE
FROM finals f
)

In [0]:
%sql
-- Bootstrap: create the SCD2 target only if missing. On nightly job runs this is
-- a no-op — the three SCD2 steps below do all the work.
CREATE TABLE IF NOT EXISTS crm_ingestion.gold.customer_health_profile (
    LEAD_ID                             STRING NOT NULL,
    LEAD_EMAIL                          STRING,
    LEAD_NAME                           STRING,
    VRSN_START_DATE                     DATE    COMMENT 'Run date this version opened (first run = first appearance).',
    VRSN_END_DATE                       DATE    COMMENT 'Run date the hash changed and this version was superseded. NULL = current version.',
    CONTRACT_VALUE                      DOUBLE,
    CONTRACT_START_DATE                 DATE,
    DAYS_SINCE_LAST_EMAIL               INT,
    DAYS_SINCE_LAST_LOGIN               INT,
    DAYS_SINCE_LAST_MEETING             INT,
    DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT      BIGINT,
    DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER BIGINT,
    TOTAL_MISSED_CALLS                  BIGINT  COMMENT 'Missed inbound calls in the trailing 14-day window (not a lifetime total).',
    SENTIMENTS_LAST_30_DAYS             STRING,
    SLACK_ENGAGED_FLAG                  INT,
    PLATFORM_ENGAGED_FLAG               INT,
    MEETING_ENGAGED_FLAG                INT,
    ENGAGEMENT_PATTERN                  STRING,
    BASE_ENGAGEMENT_SCORE               INT     COMMENT 'From the pattern table (90, 80, 75, 60, 50, 40, 30, 20, or no-data).',
    FINAL_SCORE                        INT     COMMENT 'Base + missed-call adjustment + sentiment adjustment. No adjustments on NO DATA. Not capped.',
    HEALTH_BAND                         STRING,
    HASH_VALUE                          STRING  COMMENT 'MD5 over the 13 engagement-hash columns; a change opens a new version.',
    SOURCE_UPDATED_AT                   TIMESTAMP COMMENT 'CRM date_updated of the lead record when this version was created.',
    INSERT_DATE                         TIMESTAMP COMMENT 'When this pipeline created the version row.'
);

COMMENT ON TABLE crm_ingestion.gold.customer_health_profile IS
    'SCD2 engagement history: one row per lead per engagement version. A new version opens when the engagement hash changes; VRSN_END_DATE NULL = current. Companion to customer_health_history (daily snapshots).';

In [0]:
%sql
-- SCD2 step 1: close versions whose hash changed, refresh clock metrics on
-- unchanged open versions. Delta evaluates the MATCHED clauses in order.
MERGE INTO crm_ingestion.gold.customer_health_profile AS t
USING customer_engagement_source AS s
ON  t.LEAD_ID = s.LEAD_ID
AND t.VRSN_END_DATE IS NULL
WHEN MATCHED AND t.HASH_VALUE <> s.HASH_VALUE THEN
    UPDATE SET t.VRSN_END_DATE = CURRENT_DATE()
WHEN MATCHED THEN
    UPDATE SET
        t.CONTRACT_START_DATE                 = s.CONTRACT_START_DATE,
        t.DAYS_SINCE_LAST_EMAIL               = s.DAYS_SINCE_LAST_EMAIL,
        t.DAYS_SINCE_LAST_LOGIN               = s.DAYS_SINCE_LAST_LOGIN,
        t.DAYS_SINCE_LAST_MEETING             = s.DAYS_SINCE_LAST_MEETING,
        t.DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT = s.DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT,
        t.DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER = s.DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER,
        t.TOTAL_MISSED_CALLS                  = s.TOTAL_MISSED_CALLS,
        t.SOURCE_UPDATED_AT                   = s.SOURCE_UPDATED_AT

In [0]:
%sql
-- SCD2 step 2: open a new version for new leads and for leads whose version
-- step 1 just closed. Unchanged leads still have an open version -> skipped.
INSERT INTO crm_ingestion.gold.customer_health_profile (
    LEAD_ID, LEAD_EMAIL, LEAD_NAME, VRSN_START_DATE, VRSN_END_DATE,
    CONTRACT_VALUE, CONTRACT_START_DATE,
    DAYS_SINCE_LAST_EMAIL, DAYS_SINCE_LAST_LOGIN, DAYS_SINCE_LAST_MEETING,
    DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT, DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER,
    TOTAL_MISSED_CALLS, SENTIMENTS_LAST_30_DAYS,
    SLACK_ENGAGED_FLAG, PLATFORM_ENGAGED_FLAG, MEETING_ENGAGED_FLAG,
    ENGAGEMENT_PATTERN, BASE_ENGAGEMENT_SCORE, FINAL_SCORE, HEALTH_BAND,
    HASH_VALUE, SOURCE_UPDATED_AT, INSERT_DATE
)
SELECT
    s.LEAD_ID, s.LEAD_EMAIL, s.LEAD_NAME,
    CURRENT_DATE()                    AS VRSN_START_DATE,
    CAST(NULL AS DATE)               AS VRSN_END_DATE,
    s.CONTRACT_VALUE, s.CONTRACT_START_DATE,
    s.DAYS_SINCE_LAST_EMAIL, s.DAYS_SINCE_LAST_LOGIN, s.DAYS_SINCE_LAST_MEETING,
    s.DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT, s.DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER,
    s.TOTAL_MISSED_CALLS, s.SENTIMENTS_LAST_30_DAYS,
    s.SLACK_ENGAGED_FLAG, s.PLATFORM_ENGAGED_FLAG, s.MEETING_ENGAGED_FLAG,
    s.ENGAGEMENT_PATTERN, s.BASE_ENGAGEMENT_SCORE, s.FINAL_SCORE, s.HEALTH_BAND,
    s.HASH_VALUE, s.SOURCE_UPDATED_AT,
    CURRENT_TIMESTAMP()               AS INSERT_DATE
FROM customer_engagement_source s
WHERE NOT EXISTS (
    SELECT 1
    FROM crm_ingestion.gold.customer_health_profile v
    WHERE v.LEAD_ID = s.LEAD_ID
      AND v.VRSN_END_DATE IS NULL
)

In [0]:
%sql
-- SCD2 step 3: a lead that left the Coaching Client population closes its open
-- version today. If it returns later, step 2 opens a fresh version.
UPDATE crm_ingestion.gold.customer_health_profile
SET VRSN_END_DATE = CURRENT_DATE()
WHERE VRSN_END_DATE IS NULL
  AND LEAD_ID NOT IN (SELECT LEAD_ID FROM customer_engagement_source)

In [0]:
%sql
-- Validation 1: open versions must equal the current population (1,339),
-- and every open version started on the first run date.
SELECT
    COUNT(*)            AS open_versions,
    COUNT(DISTINCT LEAD_ID) AS distinct_leads,
    MIN(VRSN_START_DATE)    AS first_version_date,
    MAX(VRSN_START_DATE)    AS latest_version_start
FROM crm_ingestion.gold.customer_health_profile
WHERE VRSN_END_DATE IS NULL

In [0]:
%sql
-- Validation 2: current engagement mix by band (open versions only)
SELECT HEALTH_BAND, ENGAGEMENT_PATTERN, COUNT(*) AS leads
FROM crm_ingestion.gold.customer_health_profile
WHERE VRSN_END_DATE IS NULL
GROUP BY 1, 2
ORDER BY leads DESC

In [0]:
%sql
-- Validation 3: version churn — leads with more than one version. Empty on the
-- first run; from tomorrow on, this is the change log.
SELECT LEAD_ID, COUNT(*) AS versions, COUNT(*) - 1 AS changes,
       MIN(VRSN_START_DATE) AS first_seen, MAX(VRSN_START_DATE) AS last_change
FROM crm_ingestion.gold.customer_health_profile
GROUP BY LEAD_ID
HAVING COUNT(*) > 1
ORDER BY changes DESC, last_change DESC
LIMIT 10

In [0]:
%sql
select count(distinct lead_id) from crm_ingestion.gold.customer_health_profile

In [0]:
%sql
SELECT
    *
FROM
    crm_ingestion.gold.customer_health_profile
LIMIT 
    10;